In [4]:
import pandas as pd
import email
from email import policy
from email.parser import BytesParser



In [68]:
import_directory = r'C:\Users\timur\Documents\GitHub\EmailSentin\data\bodies.csv'

In [40]:
# Display the DataFrame

def extract_email_content(raw_email) -> dict:
    """
    Extracts content from a raw email message.

    Parameters:
    - raw_email: Raw email data (bytes or string).

    Returns:
    - A dictionary containing the email's subject, date, from, to, and body content.
    """
    if isinstance(raw_email, str):
        # If raw_email is a string, convert it to bytes
        raw_email = raw_email.encode('utf-8')
    
    # Parse the raw email
    msg = BytesParser(policy=policy.default).parsebytes(raw_email)
    
    # Extract email details
    email_content = {
        'Subject': msg['subject'],
        'Date': msg['date'],
        'From': msg['from'],
        'To': msg['to'],
        'Body': msg.get_body(preferencelist=('plain')).get_content() if msg.is_multipart() else msg.get_content()
    }
    
    return email_content['Body']


def select_random_rows(df: pd.DataFrame, n: int) -> pd.DataFrame:
    if n > len(df):
        raise ValueError("Sample size n cannot be greater than the number of rows in the DataFrame")
    sampled_df = df.sample(n=n, random_state=1)
    return sampled_df


In [7]:
# Specify the path to your CSV file
csv_file_path = r"C:\Users\timur\Documents\GitHub\EmailSentin\data\email_enron.csv"

# Load the CSV file into a DataFrame, only loading the first 5 rows
df = pd.read_csv(csv_file_path, nrows=1000)

In [37]:
sampled_df['message'].iloc[0]

"Message-ID: <17820178.1075846925335.JavaMail.evans@thyme>\nDate: Tue, 4 Jan 2000 08:20:00 -0800 (PST)\nFrom: tana.jones@enron.com\nTo: alicia.goodrow@enron.com\nSubject: Re: Dinner\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Tana Jones\nX-To: Alicia Goodrow\nX-cc: \nX-bcc: \nX-Folder: \\Tanya_Jones_Dec2000\\Notes Folders\\All documents\nX-Origin: JONES-T\nX-FileName: tjones.nsf\n\nIt would be nice if you could be at my dinner, since I probably won't know \nanyone else.  Anytime you want to go to lunch to check on the house status, \nI'd be glad to go..."

In [38]:
sampled_df = select_random_rows(df,n=100)


sampled_df['Body'] = sampled_df['message'].apply(extract_email_body)


In [48]:
sampled_df['Body'].iloc[6]

'Kim can you also invite Mike Roberts.'

In [59]:
def check_for_thread_indicators(body):
    thread_indicators = [
        "On", "At", "----- Original Message -----", 
        "Forwarded by", "Forwarded message", 
        "Re:", "Fwd:"
    ]
    return not any(indicator in body for indicator in thread_indicators)

def extract_body(raw_email):
    msg = BytesParser(policy=policy.default).parsebytes(raw_email.encode('utf-8'))
    body = msg.get_body(preferencelist=('plain')).get_content() if msg.is_multipart() else msg.get_content()
    return body


is_nested = check_for_thread_indicators(sampled_df['Body'].iloc[6])

non_nested_df = sampled_df[sampled_df['Body'].apply(check_for_thread_indicators)]

In [66]:
body_sequence = non_nested_df['Body'].reset_index(drop=True,inplace=False)

In [77]:
body_sequence

0     It would be nice if you could be at my dinner,...
1     Christine:\n\nMy apologies.  My schedule melte...
2                 Kim can you also invite Mike Roberts.
3     If you wish to unsubscribe please CLICK HERE: ...
4     Start Date: 3/31/01; HourAhead hour: 15;  No a...
5                you still at school?  call me at work.
6     Hi Carol,\n\nI saw that you posted about getti...
7     Sorry I lost my temper.  I guess I'm not accus...
8     There will be a staff meeting tomorrow morning...
9     505434 - Killed.\n\n505066 - Changed to APB.\n...
10    \n\n\n\t9-Apr\t10-Apr\t11-Apr\t12-Apr\t13-Apr\...
11      ?\n - Doves - Lost Souls - 08 Catch the Sun.mp3
12    \nPlease see attached.\n\n  \nRegards,\n\nWend...
13    Do you know what the volumns per day are for #...
14    \nJust a reminder that when entering a wheel, ...
15    Leslie,after seeing point # 3 in writing , I w...
16    The e-mail I sent yesterday has an attachment ...
17    Please delete all messages that you receiv

In [80]:
df = body_sequence.to_frame(name='text')

# Add an empty 'label' column
df['label'] = ''

df.to_csv(import_directory, header=True)


In [79]:
df 

,text,label
0,"It would be nice if you could be at my dinner,...",
1,Christine:\n\nMy apologies. My schedule melte...,
2,Kim can you also invite Mike Roberts.,
3,If you wish to unsubscribe please CLICK HERE: ...,
4,Start Date: 3/31/01; HourAhead hour: 15; No a...,
5,you still at school? call me at work.,
6,"Hi Carol,\n\nI saw that you posted about getti...",
7,Sorry I lost my temper. I guess I'm not accus...,
8,There will be a staff meeting tomorrow morning...,
9,505434 - Killed.\n\n505066 - Changed to APB.\n...,
